# Análise exploratória | ZGLMM214 | tbl_ds_pro_zglmm214_cadastro_fornecedor

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`` |
| Ambiente | DEV |
| Objeto | Tabela |
| Colunas do DESCRIBE | 19 |
| Tipos | boolean, date, string, timestamp |
| Clustering declarado | Não informado no CSV |
| Centro | Não consta; `cod_empresa` é apenas recorte alternativo, NÃO centro |
| Chave candidata | `cod_fornecedor` + `cod_empresa` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_ZGLMM214.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | Sem cod_centro: não presumir equivalência do filtro SAP por centro. |
| 🔴 ALTA | 8 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;


### 1.1. Detalhes e histórico da tabela

Comandos independentes, executados somente em tabelas.


In [0]:
DESCRIBE DETAIL `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;


In [0]:
DESCRIBE HISTORY `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_fornecedor`) AS distintos_cod_fornecedor, COUNT(DISTINCT `cod_empresa`) AS cod_empresa_distintos_nao_centros FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;


## 3. Distribuição por centro

O schema não contém cod_centro; a consulta abaixo mostra somente um recorte alternativo, sem equivalência ao centro SAP.


In [0]:
SELECT `cod_empresa` AS recorte_alternativo_nao_centro, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_empresa` ORDER BY linhas DESC LIMIT 100;


## 3.1. Centros 4128 e 4014

Não existe campo de centro confirmado. Esta célula registra a limitação sem aplicar filtro inventado.


In [0]:
SELECT 'SEM_COLUNA_CENTRO' AS situacao, '4128 e 4014 nao podem ser medidos diretamente neste schema' AS observacao;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'cod_fornecedor' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `cod_fornecedor` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) k UNION ALL SELECT 'cod_fornecedor + cod_empresa' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `cod_fornecedor`, `cod_empresa` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) k UNION ALL SELECT 'cod_fornecedor + cod_empresa + cod_endereco' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `cod_fornecedor`, `cod_empresa`, `cod_endereco` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`cod_fornecedor`, t.`cod_empresa`, md5(to_json(struct(t.`cod_fornecedor`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_cnpj`, t.`cod_grupo_contas_fornecedor`, t.`ds_logradouro`, t.`nm_municipio`, t.`sg_estado`, t.`sg_pais`, t.`num_cep`, t.`num_telefone1`, t.`num_fax`, t.`cod_endereco`, t.`ind_excluido`, t.`ds_email`, t.`dh_carga`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` t), d AS (SELECT `cod_fornecedor`, `cod_empresa`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `cod_fornecedor`, `cod_empresa` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `cod_fornecedor`, `cod_empresa`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`cod_fornecedor` AS STRING)) AS `dist_cod_fornecedor`, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS `dist_cod_empresa`, COUNT(DISTINCT CAST(`cod_grupo_contas_fornecedor` AS STRING)) AS `dist_cod_grupo_contas_fornecedor`, COUNT(DISTINCT CAST(`nm_municipio` AS STRING)) AS `dist_nm_municipio`, COUNT(DISTINCT CAST(`sg_estado` AS STRING)) AS `dist_sg_estado`, COUNT(DISTINCT CAST(`sg_pais` AS STRING)) AS `dist_sg_pais`, COUNT(DISTINCT CAST(`ind_excluido` AS STRING)) AS `dist_ind_excluido`, COUNT(DISTINCT CAST(`dh_carga` AS STRING)) AS `dist_dh_carga`, COUNT(DISTINCT CAST(`dateingest` AS STRING)) AS `dist_dateingest`, COUNT(DISTINCT CAST(`yearingest` AS STRING)) AS `dist_yearingest`, COUNT(DISTINCT CAST(`monthingest` AS STRING)) AS `dist_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_fornecedor`, `cod_empresa` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_cod_fornecedor` > 1) AS `chaves_variam_cod_fornecedor`, COUNT_IF(`dist_cod_empresa` > 1) AS `chaves_variam_cod_empresa`, COUNT_IF(`dist_cod_grupo_contas_fornecedor` > 1) AS `chaves_variam_cod_grupo_contas_fornecedor`, COUNT_IF(`dist_nm_municipio` > 1) AS `chaves_variam_nm_municipio`, COUNT_IF(`dist_sg_estado` > 1) AS `chaves_variam_sg_estado`, COUNT_IF(`dist_sg_pais` > 1) AS `chaves_variam_sg_pais`, COUNT_IF(`dist_ind_excluido` > 1) AS `chaves_variam_ind_excluido`, COUNT_IF(`dist_dh_carga` > 1) AS `chaves_variam_dh_carga`, COUNT_IF(`dist_dateingest` > 1) AS `chaves_variam_dateingest`, COUNT_IF(`dist_yearingest` > 1) AS `chaves_variam_yearingest`, COUNT_IF(`dist_monthingest` > 1) AS `chaves_variam_monthingest` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(19, 'cod_fornecedor', 'string', c0_n, c0_v, c0_z, 'cod_empresa', 'string', c1_n, c1_v, c1_z, 'nm_fornecedor', 'string', c2_n, c2_v, c2_z, 'num_cnpj', 'string', c3_n, c3_v, c3_z, 'cod_grupo_contas_fornecedor', 'string', c4_n, c4_v, c4_z, 'ds_logradouro', 'string', c5_n, c5_v, c5_z, 'nm_municipio', 'string', c6_n, c6_v, c6_z, 'sg_estado', 'string', c7_n, c7_v, c7_z, 'sg_pais', 'string', c8_n, c8_v, c8_z, 'num_cep', 'string', c9_n, c9_v, c9_z, 'num_telefone1', 'string', c10_n, c10_v, c10_z, 'num_fax', 'string', c11_n, c11_v, c11_z, 'cod_endereco', 'string', c12_n, c12_v, c12_z, 'ind_excluido', 'boolean', c13_n, c13_v, c13_z, 'ds_email', 'string', c14_n, c14_v, c14_z, 'dh_carga', 'timestamp', c15_n, c15_v, c15_z, 'dateingest', 'date', c16_n, c16_v, c16_z, 'yearingest', 'string', c17_n, c17_v, c17_z, 'monthingest', 'string', c18_n, c18_v, c18_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`cod_fornecedor` IS NULL) AS c0_n, COUNT_IF(`cod_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`cod_empresa` IS NULL) AS c1_n, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`nm_fornecedor` IS NULL) AS c2_n, COUNT_IF(`nm_fornecedor` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`num_cnpj` IS NULL) AS c3_n, COUNT_IF(`num_cnpj` IS NOT NULL AND LOWER(TRIM(`num_cnpj`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c3_v, COUNT_IF(TRIM(`num_cnpj`) RLIKE '^0+([.,]0+)?$') AS c3_z, COUNT_IF(`cod_grupo_contas_fornecedor` IS NULL) AS c4_n, COUNT_IF(`cod_grupo_contas_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_grupo_contas_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c4_v, COUNT_IF(TRIM(`cod_grupo_contas_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c4_z, COUNT_IF(`ds_logradouro` IS NULL) AS c5_n, COUNT_IF(`ds_logradouro` IS NOT NULL AND LOWER(TRIM(`ds_logradouro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`ds_logradouro`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`nm_municipio` IS NULL) AS c6_n, COUNT_IF(`nm_municipio` IS NOT NULL AND LOWER(TRIM(`nm_municipio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c6_v, COUNT_IF(TRIM(`nm_municipio`) RLIKE '^0+([.,]0+)?$') AS c6_z, COUNT_IF(`sg_estado` IS NULL) AS c7_n, COUNT_IF(`sg_estado` IS NOT NULL AND LOWER(TRIM(`sg_estado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c7_v, COUNT_IF(TRIM(`sg_estado`) RLIKE '^0+([.,]0+)?$') AS c7_z, COUNT_IF(`sg_pais` IS NULL) AS c8_n, COUNT_IF(`sg_pais` IS NOT NULL AND LOWER(TRIM(`sg_pais`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c8_v, COUNT_IF(TRIM(`sg_pais`) RLIKE '^0+([.,]0+)?$') AS c8_z, COUNT_IF(`num_cep` IS NULL) AS c9_n, COUNT_IF(`num_cep` IS NOT NULL AND LOWER(TRIM(`num_cep`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c9_v, COUNT_IF(TRIM(`num_cep`) RLIKE '^0+([.,]0+)?$') AS c9_z, COUNT_IF(`num_telefone1` IS NULL) AS c10_n, COUNT_IF(`num_telefone1` IS NOT NULL AND LOWER(TRIM(`num_telefone1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`num_telefone1`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`num_fax` IS NULL) AS c11_n, COUNT_IF(`num_fax` IS NOT NULL AND LOWER(TRIM(`num_fax`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`num_fax`) RLIKE '^0+([.,]0+)?$') AS c11_z, COUNT_IF(`cod_endereco` IS NULL) AS c12_n, COUNT_IF(`cod_endereco` IS NOT NULL AND LOWER(TRIM(`cod_endereco`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c12_v, COUNT_IF(TRIM(`cod_endereco`) RLIKE '^0+([.,]0+)?$') AS c12_z, COUNT_IF(`ind_excluido` IS NULL) AS c13_n, CAST(0 AS BIGINT) AS c13_v, CAST(0 AS BIGINT) AS c13_z, COUNT_IF(`ds_email` IS NULL) AS c14_n, COUNT_IF(`ds_email` IS NOT NULL AND LOWER(TRIM(`ds_email`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c14_v, COUNT_IF(TRIM(`ds_email`) RLIKE '^0+([.,]0+)?$') AS c14_z, COUNT_IF(`dh_carga` IS NULL) AS c15_n, CAST(0 AS BIGINT) AS c15_v, CAST(0 AS BIGINT) AS c15_z, COUNT_IF(`dateingest` IS NULL) AS c16_n, CAST(0 AS BIGINT) AS c16_v, CAST(0 AS BIGINT) AS c16_z, COUNT_IF(`yearingest` IS NULL) AS c17_n, COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c17_v, COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS c17_z, COUNT_IF(`monthingest` IS NULL) AS c18_n, COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c18_v, COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS c18_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte

Não há coluna explicitamente classificada como sem fonte no DESCRIBE; não conclui que todas são preenchidas.


In [0]:
SELECT 'Nenhuma coluna explicitamente classificada como sem fonte no DESCRIBE' AS observacao;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `cod_fornecedor`) AS d0, COUNT(DISTINCT `cod_empresa`) AS d1, COUNT(DISTINCT `cod_grupo_contas_fornecedor`) AS d2, COUNT(DISTINCT `sg_estado`) AS d3, COUNT(DISTINCT `sg_pais`) AS d4, COUNT(DISTINCT `ind_excluido`) AS d5 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(6, 'cod_fornecedor', d0, 'cod_empresa', d1, 'cod_grupo_contas_fornecedor', d2, 'sg_estado', d3, 'sg_pais', d4, 'ind_excluido', d5) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_empresa` UNION ALL SELECT 'cod_grupo_contas_fornecedor' AS coluna, CAST(`cod_grupo_contas_fornecedor` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_grupo_contas_fornecedor` UNION ALL SELECT 'sg_estado' AS coluna, CAST(`sg_estado` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `sg_estado` UNION ALL SELECT 'sg_pais' AS coluna, CAST(`sg_pais` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `sg_pais` UNION ALL SELECT 'ind_excluido' AS coluna, CAST(`ind_excluido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `ind_excluido`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

Mede os estados observados, sem converter indicadores STRING em booleanos por suposição.


In [0]:
WITH a AS (SELECT 'ind_excluido' AS coluna, CAST(`ind_excluido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `ind_excluido`) SELECT coluna, valor, linhas FROM a ORDER BY coluna, linhas DESC;


## 10. Perfil numérico

Não há quantidades ou valores numéricos declarados neste schema.


In [0]:
SELECT 'Sem valores ou quantidades numericos para perfilar' AS observacao;


## 11. Totais para conciliação

Não há quantidades ou valores numéricos declarados neste schema.


In [0]:
SELECT 'Sem totais monetarios ou quantitativos numericos' AS observacao;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dh_carga' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dh_carga` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dh_carga` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dh_carga`) AS STRING) AS menor_literal, CAST(MAX(`dh_carga`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` UNION ALL SELECT 'dateingest' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dateingest` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dateingest`) AS STRING) AS menor_literal, CAST(MAX(`dateingest`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'cod_fornecedor' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_fornecedor`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_fornecedor`))) AS tamanho_max, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_fornecedor`) RLIKE '^[0-9]+$' THEN TRIM(`cod_fornecedor`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_fornecedor`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_fornecedor`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_empresa`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_empresa`))) AS tamanho_max, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN TRIM(`cod_empresa`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_empresa`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` UNION ALL SELECT 'cod_grupo_contas_fornecedor' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_grupo_contas_fornecedor`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_grupo_contas_fornecedor`))) AS tamanho_max, COUNT_IF(TRIM(`cod_grupo_contas_fornecedor`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_grupo_contas_fornecedor`) RLIKE '^[0-9]+$' THEN TRIM(`cod_grupo_contas_fornecedor`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_grupo_contas_fornecedor`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_grupo_contas_fornecedor`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`cod_fornecedor`, t.`cod_empresa`, t.`nm_fornecedor`, t.`num_cnpj`, t.`cod_grupo_contas_fornecedor`, t.`ds_logradouro`, t.`nm_municipio`, t.`sg_estado`, t.`sg_pais`, t.`num_cep`, t.`num_telefone1`, t.`num_fax`, t.`cod_endereco`, t.`ind_excluido`, t.`ds_email`, t.`dh_carga`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `cod_fornecedor`, `cod_empresa`, `cod_grupo_contas_fornecedor`, `nm_municipio`, `sg_estado`, `sg_pais`, `ind_excluido`, `dh_carga`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`num_cnpj` AS STRING)),1,8) AS `num_cnpj_hash`, SUBSTRING(MD5(CAST(`ds_logradouro` AS STRING)),1,8) AS `ds_logradouro_hash`, SUBSTRING(MD5(CAST(`num_cep` AS STRING)),1,8) AS `num_cep_hash`, SUBSTRING(MD5(CAST(`num_telefone1` AS STRING)),1,8) AS `num_telefone1_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_empresa`, `cod_grupo_contas_fornecedor`, `sg_estado`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_empresa`, `cod_grupo_contas_fornecedor`, `sg_estado` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Mede datas de carga e partições; não confundir carimbo de ingestão com evento de negócio.


In [0]:
SELECT COUNT(*) AS linhas, MIN(`dateingest`) AS `min_dateingest`, MAX(`dateingest`) AS `max_dateingest`, COUNT(DISTINCT `dateingest`) AS `distintos_dateingest`, MIN(`dh_carga`) AS `min_dh_carga`, MAX(`dh_carga`) AS `max_dh_carga`, COUNT(DISTINCT `dh_carga`) AS `distintos_dh_carga`, MIN(`yearingest`) AS `min_yearingest`, MAX(`yearingest`) AS `max_yearingest`, COUNT(DISTINCT `yearingest`) AS `distintos_yearingest`, MIN(`monthingest`) AS `min_monthingest`, MAX(`monthingest`) AS `max_monthingest`, COUNT(DISTINCT `monthingest`) AS `distintos_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Cadastro, empresa e exclusão


In [0]:
SELECT `cod_empresa` AS empresa, `ind_excluido` AS excluido, COUNT(*) AS linhas, COUNT(DISTINCT `cod_fornecedor`) AS fornecedores, COUNT_IF(`ds_email` IS NOT NULL AND TRIM(`ds_email`) <> '') AS emails_preenchidos, COUNT_IF(`num_cnpj` IS NOT NULL AND TRIM(`num_cnpj`) <> '') AS documentos_fiscais_preenchidos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_empresa`, `ind_excluido` ORDER BY linhas DESC;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `cod_fornecedor`, `cod_empresa`, `cod_grupo_contas_fornecedor`, `nm_municipio`, `sg_estado`, `sg_pais`, `ind_excluido`, `dh_carga`, `dateingest`, `yearingest`, `monthingest`, SUBSTRING(MD5(CAST(`nm_fornecedor` AS STRING)),1,8) AS `nm_fornecedor_hash`, SUBSTRING(MD5(CAST(`num_cnpj` AS STRING)),1,8) AS `num_cnpj_hash`, SUBSTRING(MD5(CAST(`ds_logradouro` AS STRING)),1,8) AS `ds_logradouro_hash`, SUBSTRING(MD5(CAST(`num_cep` AS STRING)),1,8) AS `num_cep_hash`, SUBSTRING(MD5(CAST(`num_telefone1` AS STRING)),1,8) AS `num_telefone1_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` LIMIT 500;


## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `cod_fornecedor`, `cod_empresa`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor` GROUP BY `cod_fornecedor`, `cod_empresa`), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zglmm214_cadastro_fornecedor`) SELECT 'ZGLMM214' AS transacao, 'tbl_ds_pro_zglmm214_cadastro_fornecedor' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, 'SEM CENTRO NO SCHEMA' AS filtro_centro, 'Chave candidata: cod_fornecedor + cod_empresa' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
